Module 1

In [1]:
%pip install pandas numpy scikit-learn joblib xgboost

  Using cached pandas-3.0.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.9.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached xgboost-3.3.0-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached pandas-3.0.3-cp313-cp313-win_amd64.whl (9.8 MB)
Using cached scikit_learn-1.9.0-cp313-cp313-win_amd64.whl (8.2 MB)
Using cached xgboost-3.3.0-py3-none-win_amd64.whl (69.5 MB)

   ---------------------------------------- 0/3 [xgboost]
   ---------------------------------------- 0/3 [xgboost]
   ---------------------------------------- 0/3 [xgboost]
   ---------------------------------------- 0/3 [xgboost]
   ---------------------------------------- 0/3 [xgboost]
   ---------------------------------------- 0/3 [xgboost]
   ---------------------------------------- 0/3 [xgboost]
   ---------------------------------------- 0/3 [xgboost]
   ---------------------------------------- 0/3 [xgboost]
   ---------------------------------------- 0/3 [xgboost]
   ----------

Import libraries

In [2]:
import os
import pandas as pd
import numpy as np

set the datapath

In [6]:
BASE_DIR = os.getcwd()

DATASET_PATH = os.path.join(BASE_DIR, "data", "dataset", "employee_wellness.csv")
PROCESSED_DATA_PATH = os.path.join(BASE_DIR, "data", "processed", "cleaned_employee_wellness.csv")
MODEL_PATH = os.path.join(BASE_DIR, "models", "wellness_risk_model.pkl")

print(DATASET_PATH)
print(PROCESSED_DATA_PATH)
print(MODEL_PATH)

c:\Users\Lenovo\OneDrive\Documents\Infosys_Virtual_internship\Employee_Wellness_analytics\backend\src\data\dataset\employee_wellness.csv
c:\Users\Lenovo\OneDrive\Documents\Infosys_Virtual_internship\Employee_Wellness_analytics\backend\src\data\processed\cleaned_employee_wellness.csv
c:\Users\Lenovo\OneDrive\Documents\Infosys_Virtual_internship\Employee_Wellness_analytics\backend\src\models\wellness_risk_model.pkl


Load the dataset

In [7]:
df = pd.read_csv(DATASET_PATH)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Lenovo\\OneDrive\\Documents\\Infosys_Virtual_internship\\Employee_Wellness_analytics\\backend\\src\\data\\dataset\\employee_wellness.csv'

Validate required columns

In [ ]:
required_columns = [
    "employeeid",
    "age",
    "gender",
    "heightcm",
    "weightkg",
    "bmi",
    "sleephours",
    "exercisedaysperweek",
    "stresslevel",
    "attendancepercent",
    "medicalcondition",
    "stressscore",
    "smoker",
    "alcoholuse",
    "bloodpressuresystolic",
    "bloodpressurediastolic",
    "glucoselevel",
    "risklabel"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are present.")

Inspect dataset info

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

Copy dataset before cleaning

In [ ]:
cleaned_df = df.copy()

In [ ]:
numeric_columns = [
    "age",
    "heightcm",
    "weightkg",
    "bmi",
    "sleephours",
    "exercisedaysperweek",
    "stresslevel",
    "attendancepercent",
    "stressscore",
    "bloodpressuresystolic",
    "bloodpressurediastolic",
    "glucoselevel"
]

for col in numeric_columns:
    cleaned_df[col] = pd.to_numeric(cleaned_df[col], errors="coerce")
    cleaned_df[col] = cleaned_df[col].fillna(cleaned_df[col].median())

remove duplicates

In [ ]:
cleaned_df = cleaned_df.drop_duplicates()
print("Final row count after cleaning:", len(cleaned_df))

preview processed data

In [ ]:
cleaned_df.head()

Module 2

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import joblib

In [ ]:
model_df = pd.read_csv(PROCESSED_DATA_PATH)
model_df.head()

In [ ]:
label_encoder = LabelEncoder()
model_df["burnout_risk_encoded"] = label_encoder.fit_transform(model_df["burnout_risk"])

print("Target classes:", list(label_encoder.classes_))

In [ ]:
model_df_encoded = pd.get_dummies(
    model_df,
    columns=["gender", "medical_condition"],
    drop_first=True
)

model_df_encoded.head()

In [ ]:
X = model_df_encoded.drop(columns=["employee_id", "burnout_risk", "burnout_risk_encoded"])
y = model_df_encoded["burnout_risk_encoded"]

print("Feature columns:")
print(X.columns.tolist())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Train Random Forest model

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

predict on test set

In [ ]:
y_pred = rf_model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
joblib.dump(rf_model, MODEL_PATH)
joblib.dump(label_encoder, os.path.join(BASE_DIR, "models", "label_encoder.pkl"))
joblib.dump(X.columns.tolist(), os.path.join(BASE_DIR, "models", "feature_columns.pkl"))

print("Model saved to:", MODEL_PATH)
print("Label encoder saved.")
print("Feature columns saved.")

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance.head(10)

predict one sample employee

In [ ]:
sample_input = {
    "age": 35,
    "height_cm": 170,
    "weight_kg": 78,
    "sleep_hours": 5,
    "exercise_days_per_week": 1,
    "stress_level": 8,
    "attendance_percent": 82,
    "bmi": 78 / ((170 / 100) ** 2),
    "gender": "Male",
    "medical_condition": "Hypertension"
}

sample_df = pd.DataFrame([sample_input])
sample_df = pd.get_dummies(sample_df)

saved_columns = joblib.load(os.path.join(BASE_DIR, "models", "feature_columns.pkl"))
sample_df = sample_df.reindex(columns=saved_columns, fill_value=0)

loaded_model = joblib.load(MODEL_PATH)
loaded_label_encoder = joblib.load(os.path.join(BASE_DIR, "models", "label_encoder.pkl"))

prediction = loaded_model.predict(sample_df)
predicted_label = loaded_label_encoder.inverse_transform(prediction)

print("Predicted wellness risk:", predicted_label[0])